In [2]:
#@markdown Please fill in the value below with your GCP project ID and then run the cell.
PROJECT_ID = "leafy-guide-482019-n1"

# Quick input validation
assert PROJECT_ID, "⚠ Please provide your Google Cloud Project ID"

# Configure gcloud.
!gcloud config set project {PROJECT_ID}
print(f'Project has been set to {PROJECT_ID}')

Project has been set to leafy-guide-482019-n1



To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].


In [3]:
# Authentication step

"""Colab Auth""" 
# from google.colab import auth
# auth.authenticate_user()


"""Jupiter Notebook Auth"""
import google.auth
import os

credentials, project_id = google.auth.default()

os.environ['GOOGLE_CLOUD_QUOTA_PROJECT']=PROJECT_ID
os.environ['GOOGLE_CLOUD_PROJECT']=PROJECT_ID

In [6]:
#Enable all the required APIs for the COPY

!gcloud services enable cloudapis.googleapis.com compute.googleapis.com iam.googleapis.com bigquery.googleapis.com --project {PROJECT_ID}

Operation "operations/acf.p2-264862545614-c4b212ad-a3ea-4ba1-968e-5130ad29aa92" finished successfully.
Operation "operations/acat.p2-264862545614-2b65a75e-d8fc-4cbd-8e12-b409e6af698a" finished successfully.


Operation "operations/acf.p2-264862545614-0b814d13-7070-4e38-ab03-7e9de25eda91" finished successfully.


In [7]:
# Details of source Dataset
BQ_SRC_PROJECT = "bigquery-public-data"
BQ_SRC_DATASETS = ["imdb", "imdb"]
#BQ_SRC_TABLES_LIST = [["title_principals","title_crew", "title_basics","name_basics"], ["reviews", "title_ratings"]] # [] Specify empty list to copy 'all' tables, or a Specific list, eg: ["table1", "table3", "table10"]
BQ_SRC_TABLES_LIST = [["title_principals","name_basics"], ["reviews"]] # [] Specify empty list to copy 'all' tables, or a Specific list, eg: ["table1", "table3", "table10"]
BQ_SRC_REGIONS = ["us", "us"]

# Details of destination Dataset
BQ_DST_PROJECT = PROJECT_ID
BQ_DST_DATASETS =[ "imdb_people", "imdb_ratings"] # List of destinaion dataset names
BQ_DST_REGIONS = BQ_SRC_REGIONS # Change if needed

In [8]:
def createBQDataset(bq_project_id, dataset_name,dataset_region):
    from google.cloud import bigquery
    import google.api_core 

    client=bigquery.Client(project=PROJECT_ID)

    dataset_ref = f"{bq_project_id}.{dataset_name}"
    

    try:
        client.get_dataset(dataset_ref)
        print("Destination Dataset exists")
    except google.api_core.exceptions.NotFound:
        print("Cannot find the dataset hence creating.......")
        dataset=bigquery.Dataset(dataset_ref)
        dataset.location=dataset_region
        client.create_dataset(dataset)
        
    return dataset_ref

def createBQTable(bq_project_id,dataset_name, table_name):
        from google.cloud import bigquery
        import google.api_core 

        client=bigquery.Client(project=PROJECT_ID)

        table_ref = client.dataset(dataset_name, project=bq_project_id).table(table_name)

        try:
            client.get_table(table_ref)
            print(f"Destination Table {table_ref} exists")
            
        except google.api_core.exceptions.NotFound:
            print(f"Creating the table {table_ref}.......")
            table = bigquery.Table(table_ref)
            client.create_table(table)

        return table_ref

In [9]:
#Create destination table and copy table data
from google.cloud import bigquery

# Initialize BQ client
client=bigquery.Client(project=PROJECT_ID)

for BQ_SRC_DATASET, BQ_SRC_TABLES, BQ_SRC_REGION, BQ_DST_DATASET, BQ_DST_REGION, in zip(BQ_SRC_DATASETS, BQ_SRC_TABLES_LIST, BQ_SRC_REGIONS, BQ_DST_DATASETS, BQ_DST_REGIONS):
    
    # Create Destination Dataset (If the dataset already exists, delete the dataset (and the tables with in) and create an empty dataset)
    dst_dataset_ref=createBQDataset(BQ_DST_PROJECT,BQ_DST_DATASET,BQ_DST_REGION)

    if not BQ_SRC_TABLES:
        #if tables are not explicitly provided, get the list of tables from bigquery
        dataset_id = f'{BQ_SRC_PROJECT}.{BQ_SRC_DATASET}'
        bq_tables_obj = client.list_tables(dataset_id)
        BQ_SRC_TABLES = [table_obj.table_id for table_obj in bq_tables_obj]
    
    for BQ_SRC_TABLE in BQ_SRC_TABLES:

        dst_table_ref=createBQTable(BQ_DST_PROJECT,BQ_DST_DATASET,BQ_SRC_TABLE)
        #src_table_ref = client.dataset(BQ_SRC_DATASET, project=BQ_SRC_PROJECT).table(BQ_SRC_TABLE)

        src_table_full = f"{BQ_SRC_PROJECT}.{BQ_SRC_DATASET}.{BQ_SRC_TABLE}"

        # below is the custom code 
        # Query to select only first 10,000 rows
        query = f"""
            SELECT *
            FROM {src_table_full}
            LIMIT 1000
        """
        
        job_config = bigquery.QueryJobConfig(destination=dst_table_ref,
                                            write_disposition="WRITE_TRUNCATE")
        query_job = client.query(query, job_config=job_config)

        #job_config = bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")

        #copy_job = client.copy_table(src_table_ref, dst_table_ref, job_config=job_config)
        # Wait for the job to complete and check for errors
        query_job.result()  

print('Done!')

Cannot find the dataset hence creating.......
Creating the table leafy-guide-482019-n1.imdb_people.title_principals.......
Creating the table leafy-guide-482019-n1.imdb_people.name_basics.......
Cannot find the dataset hence creating.......
Creating the table leafy-guide-482019-n1.imdb_ratings.reviews.......
Done!
